In [1]:
import requests

# UNEG repository member publications
r = requests.get("https://www.unevaluation.org/repository/member-publications?tab=2") 

In [ ]:
# export the database in RIS format
import json
import rispy

# function to load and map JSON to RIS format
def load_and_map(json_filename):
    with open(json_filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    mapped_records = []
    for record in data:
        ris_record = {
            'primary_title': record.get('title'),
            'id': record.get('item-ID'),
            'first_authors': [record.get('agency')] if record.get('agency') else []
        }
        mapped_records.append(ris_record)
    return mapped_records

# read and map both files
records_1 = load_and_map('member-publications-data_2025-07-31.json')
records_2 = load_and_map('member-publications-data_2025-08-04.json')
all_records = records_1 + records_2

# save a combined RIS file
with open('member-publications-data.ris', 'w', encoding='utf-8') as f:
    rispy.dump(all_records, f)

print(f"Combined {len(all_records)} records into member-publications-data.ris")
# check this for duplicates -- seems like there are some duplicates after import into EPPI reviewer

Combined 15570 records into member-publications-data.ris


: 

In [2]:
# get all the links displayed on this page
from bs4 import BeautifulSoup
soup = BeautifulSoup(r.content)
links = soup.select("a.listing-title-link")
print(len(links)) # 18 documents per page 

18


In [ ]:
# get all the links on this page by scrolling down and loading new items
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException, StaleElementReferenceException, ElementClickInterceptedException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

from bs4 import BeautifulSoup
import time

# set up a headless browser
options = Options()
options.headless = True
driver = webdriver.Chrome(options=options)

try:
    # initialise wait after driver creation
    wait = WebDriverWait(driver, 10)

    # load the repository 
    driver.get("https://www.unevaluation.org/repository/member-publications?tab=2")

    # click "Load More" repeatedly
    while True:
        try:
            # wait for Load More button before every iteration
            load_more = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.button[rel='next']")))

            # scroll into view
            driver.execute_script("arguments[0].scrollIntoView({block: 'center', inline: 'center'});", load_more)
            time.sleep(0.5)

            # optional: wait for any loading overlay to disappear before clicking
            try:
                wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, ".loading-overlay")))
            except TimeoutException:
                pass  # if no overlay, just continue

            # try click with retries
            retries = 3
            for attempt in range(retries):
                try:
                    load_more.click()
                    break
                except ElementClickInterceptedException:
                    time.sleep(1)
            else:
                # fallback JS click if still intercepted
                driver.execute_script("arguments[0].click();", load_more)

            # wait until new items load (count increases)
            current_count = len(driver.find_elements(By.CSS_SELECTOR, "a.listing-title-link"))
            print(f"There are now {current_count} links.")
            wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "a.listing-title-link")) > current_count)

            time.sleep(0.5)

        except TimeoutException:
            print("No more 'Load More' button found, stopping.")
            break
        except StaleElementReferenceException:
            # Element became stale; loop will retry locating button
            continue
        except Exception as e:
            print("Unexpected exception:", e)
            break

    # parse loaded page
    soup = BeautifulSoup(driver.page_source, "html.parser")

finally:
    driver.quit()

# extract links
links = soup.select("a.listing-title-link")
link_data = [{"href": link.get("href"), "text": link.text.strip()} for link in links]


today = datetime.today().strftime('%Y-%m-%d')
filename = f"links_{today}.json"

# save to JSON file - the previous data will be overwritten
with open(filename, "w", encoding="utf-8") as f:
    json.dump(link_data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(link_data)} links to {filename}")

There are now 18 links.
There are now 36 links.
There are now 54 links.
There are now 72 links.
There are now 90 links.
There are now 108 links.
There are now 126 links.
There are now 144 links.
There are now 162 links.
There are now 180 links.
There are now 198 links.
There are now 216 links.
There are now 234 links.
There are now 252 links.
There are now 270 links.
There are now 288 links.
There are now 306 links.
There are now 324 links.
There are now 342 links.
There are now 360 links.
There are now 378 links.
There are now 396 links.
There are now 414 links.
There are now 432 links.
There are now 450 links.
There are now 468 links.
There are now 486 links.
There are now 504 links.
There are now 522 links.
There are now 540 links.
There are now 558 links.
There are now 576 links.
There are now 594 links.
There are now 612 links.
There are now 630 links.
There are now 648 links.
There are now 666 links.
There are now 684 links.
There are now 702 links.
There are now 720 links.
There

In [2]:
# add links where missing
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
from datetime import datetime

# set up a headless browser
options = Options()
options.headless = True
driver = webdriver.Chrome(options=options)

# file setup
today = datetime.today().strftime('%Y-%m-%d')
filename = f"links_{today}.json"
seen_hrefs = set()

# load existing data if file exists
if os.path.exists("links.json"):
    with open("links.json", "r", encoding="utf-8") as f:
        existing_links = json.load(f)
        for item in existing_links:
            seen_hrefs.add(item["href"])
else:
    existing_links = []

try:
    wait = WebDriverWait(driver, 10)
    driver.get("https://www.unevaluation.org/repository/member-publications?tab=2")

    previous_count = 0

    while True:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        new_links = soup.select("a.listing-title-link")

        for link in new_links:
            href = link.get("href")
            text = link.text.strip()
            if href and href not in seen_hrefs:
                link_entry = {"href": href, "text": text}
                existing_links.append(link_entry)
                seen_hrefs.add(href)
                with open(filename, "w", encoding="utf-8") as f:
                    json.dump(existing_links, f, indent=2, ensure_ascii=False)
                print(f"Added: {text}")

        # check if new links were added
        current_count = len(seen_hrefs)
        if current_count == previous_count:
            print("No new links found after last 'Load More'. Stopping.")
            break
        previous_count = current_count

        try:
            load_more = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.button[rel='next']")))
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more)
            time.sleep(0.5)

            try:
                load_more.click()
            except Exception:
                driver.execute_script("arguments[0].click();", load_more)

            time.sleep(2)
        except TimeoutException:
            print("No more 'Load More' button found.")
            break

finally:
    driver.quit()

print(f"Saved {len(existing_links)} links to {filename}")

No new links found after last 'Load More'. Stopping.
Saved 14320 links to links_2025-08-15.json


In [3]:
import pandas as pd

df_links = pd.read_json("links.json")

# some of the evaluations have non-unique names 
# assign them an ID similar to the node from the repository
df_links['item-ID'] = df_links['href'].str.split('/').str[2]

df_links.head()

,href,text,item-ID
0,/node/11673,Action against child trafficking for labour an...,11673
1,/node/13752,Thematic evaluation on IFAD's support to gende...,13752
2,/node/15075,Terminal Evaluation of the UNEP Project “Secre...,15075
3,/node/11634,Evaluation of WFP's Environmental Policy,11634
4,/node/15542,Evaluation of Global Health Days 2019-2023,15542


In [4]:
import os
import json

def save_data_chunk(data_chunk, output_file):
    # check if the file exists and read existing data if it does
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            try:
                existing_data = json.load(f)
            except json.JSONDecodeError:
                existing_data = []
    else:
        existing_data = []

    # append new data
    existing_data.extend(data_chunk)

    # write the combined data back to the file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(existing_data, f, ensure_ascii=False, indent=4)

In [7]:
# create a dataframe of evaluations
# adjust this to read links from the json file

import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
from datetime import datetime

base_url = "https://www.unevaluation.org"
# repo_url = "https://www.unevaluation.org/repository/member-publications?tab=2"

# get the links to individual documents
links = df_links['href']

data = []
chunk_size = 250
today = datetime.today().strftime('%Y-%m-%d')
output_file = f"member-publications-data_{today}.json"

for i, href in enumerate(links[6500:], start=6501):
    if href:
        node_url = base_url + href
        print(f"[{i}] Found node URL: {node_url}")
        redirected_r = requests.get(node_url, allow_redirects=True)
        final_url = redirected_r.url  # this is the actual document page

        print("Redirected to:", final_url)

        # parse the document page
        doc_soup = BeautifulSoup(redirected_r.content, "html.parser")

        entry = {
            "title": doc_soup.select("div.detail-head-title")[0].text if doc_soup.select("div.detail-head-title") else "NA",
            "agency": doc_soup.select("div.detail-head-items")[0].text.replace("\n", "") if doc_soup.select("div.detail-head-items") else "NA",
            "link": final_url,
           "download-link": ";".join(dw_link['href'] for dw_link in doc_soup.select("a.external-popup-link.doc-link") if dw_link.has_attr('href')) or "NA",
            "type": (e := doc_soup.select_one("td.td-label:contains('Type') + td span")) and e.text or "NA",
            "themes": "; ".join(span.text for span in doc_soup.select("td.td-label:contains('Theme') + td span")) or "NA",
            "geo-scope": (e := doc_soup.select_one("td.td-label:contains('Geographic Scope') + td")) and e.text or "NA",
            "countries": "; ".join(span.text for span in doc_soup.select("td.td-label:contains('Country') + td span")) or "NA",
            "SDGs": "; ".join(span.text for span in doc_soup.select("td.td-label:contains('SDG') + td span")) or "NA",
            "item-ID": href.split("/")[2] or "NA"
            }
        data.append(entry)

        if i % chunk_size == 0 or i == len(links):
            print(f"\nSaving chunk of {len(data)} entries to JSON")

            save_data_chunk(data, output_file)

            print("Last 3 entries snapshot:")
            for snapshot in data[-3:]:
                print(json.dumps(snapshot, indent=2, ensure_ascii=False))

            print(f"Data saved at index {i}\n")
            data = []

        time.sleep(0.2)

[6501] Found node URL: https://www.unevaluation.org/node/6845
Redirected to: https://www.unevaluation.org/member_publications/midterm-review-report-sustainable-management-highly-migratory-fish-stocks-west
[6502] Found node URL: https://www.unevaluation.org/node/6846
Redirected to: https://www.unevaluation.org/member_publications/mid-term-evaluation-project-promotion-and-scaling-climate-resilient-resource
[6503] Found node URL: https://www.unevaluation.org/node/6847
Redirected to: https://www.unevaluation.org/member_publications/terminal-evaluation-undpgef-project-transforming-market-efficient-lighting
[6504] Found node URL: https://www.unevaluation.org/node/6848
Redirected to: https://www.unevaluation.org/member_publications/automation-votersapos-identification-process-during-electoral-cycle-2015-2017
[6505] Found node URL: https://www.unevaluation.org/node/6849
Redirected to: https://www.unevaluation.org/member_publications/improving-coverage-and-management-effectiveness-protected-are

In [ ]:
import json
import pandas as pd

# load the evaluations dataset
# df has the columns title, agency, link, download-link, type, themes, geo-scope, countries, SDGs, item-ID
df = pd.read_json('member-publications-data_2025-07-31.json')
df.head()

#df1 = pd.read_json('member-publications-data_2025-08-04.json')
# df1.head()

# df = pd.concat([df, df1], ignore_index=True)

,title,agency,link,download-link,type,themes,geo-scope,countries,SDGs,item-ID
0,Action against child trafficking for labour an...,International Labour Organization,https://www.unevaluation.org/member_publicatio...,https://webapps.ilo.org/ievaldiscovery/files/f...,Project/Programme,Children; Conflict,Country,Peru,NA,11673
1,Thematic evaluation on IFAD's support to gende...,International Fund for Agricultural Development,https://www.unevaluation.org/member_publicatio...,https://ioe.ifad.org/en/w/thematic-evaluation-...,Global/Thematic,Gender,NA,NA,NA,13752
2,Terminal Evaluation of the UNEP Project “Secre...,United Nations Environment Programme,https://www.unevaluation.org/member_publicatio...,https://wedocs.unep.org/20.500.11822/47207,Project/Programme,NA,Global,NA,NA,15075
3,Evaluation of WFP's Environmental Policy,World Food Programme,https://www.unevaluation.org/member_publicatio...,https://www.wfp.org/publications/evaluation-wf...,Policy/Strategy,Environment,Global,NA,SDG2 - Zero Hunger,11634
4,Evaluation of Global Health Days 2019-2023,World Health Organization,https://www.unevaluation.org/member_publicatio...,NA,Global/Thematic,Communications and media,Global,NA,SDG3 - Good Health and Well-Being,15542


In [12]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import pandas as pd


# count countries
country_series = df['countries'].dropna().str.split(';')
country_series = country_series.explode().str.strip() # transform the list in each row into multiple lists

country_counts = country_series.value_counts().reset_index()
country_counts.columns = ['countries', 'count']

# load world map
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# fix name mismatches
country_name_corrections = {
    'United States': 'United States of America',
    'Russia': 'Russian Federation',
    'Vietnam': 'Viet Nam',
    'South Korea': 'Korea, Republic of',
    'North Korea': "Dem. People's Rep. Korea",
    'Iran': 'Iran (Islamic Republic of)',
    'Syria': 'Syrian Arab Republic',
    'Czech Republic': 'Czechia',
    'United Kingdom': 'United Kingdom',
    'Bolivia': 'Bolivia (Plurinational State of)',
}

country_counts['country'] = country_counts['country'].replace(country_name_corrections)

# merge with world map
world = world.merge(country_counts, how='left', left_on='name', right_on='country')
world['count'] = world['count'].fillna(0)

# set up a plot
fig = plt.figure(figsize=(20, 10))
ax = fig.add_subplot()
drop_idxs = world["continent"].isin(["Antarctica", "Seven seas (open ocean)"])
world = world.drop(world[drop_idxs].index)

world.plot(
    ax=ax,
    column='count',
    cmap='Greens',
    linewidth=0.5,
    edgecolor='black',
    legend=True,
    legend_kwds={'label': "Number of publications", 'shrink': 0.5}
)

# turn off axis ticks
ax.set_xticks([])
ax.set_yticks([])

plt.title("Countries mentioned in member publications", fontsize=16)
plt.show()

AttributeError: The geopandas.dataset has been deprecated and was removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.

In [ ]:
# download all the documents following the links from the repository

import os
import requests
import re
from urllib.parse import urlparse
import pandas as pd

# constants and set-up
output_folder = 'SDG-synthesis-coalition-tidy'
os.makedirs(output_folder, exist_ok=True)

# make a copy of the df
df_copy = df.copy()

if 'saved' not in df.columns:
    df['saved'] = 0 # new column to record n of downloads per report
df_copy['error_type'] = None  # new column to record errors (None means no error)

n_fails = 0

for index, item in enumerate(df_copy['download-link']):
    links = [l.strip() for l in item.split(";") if l.strip()]
    title = df_copy['title'].iloc[index]
    item_id = str(df_copy['item-ID'].iloc[index])
    print(f"[{index}] Processing: {title}, {item_id}")

    # track if any file is saved for this row and if there are erros
    any_saved_for_row = False
    errors_for_row = []

    for i, link in enumerate(links, start=1):
        # assume the extension is pdf by default 
        # further check after requests

        ext = ".pdf"
        parsed_url = urlparse(link)
        _, url_ext = os.path.splitext(parsed_url.path)
        if url_ext.lower() in [".pdf", ".doc", ".docx", ".zip"]:
            ext = url_ext.lower()

        # strip() to delete leading or trailing whitespaces
        # add _i.ext to keep track of multiple files related to one report
        
        #filename = f"{sanitise_filename(title)}_{i}{ext}"
        filename = f"{item_id}_{i}{ext}"
        filepath = os.path.join(output_folder, filename)

        if os.path.exists(filepath):
            print(f"File already exists, skipping: {filename}")
            any_saved_for_row = True   # count this as saved since file is already there
            continue

        # try downloading if file doesn't exist
        try:
            download_r = requests.get(link, allow_redirects=True, timeout=30)
            download_r.raise_for_status()

            content_type = download_r.headers.get("Content-Type", "").lower()
            ext = extract_ext(url_ext, content_type)

            # rebuild filename and filepath with final extension
            #filename = f"{sanitise_filename(title)}_{i}{ext}"

            filename = f"{item_id}_{i}{ext}"
            filepath = os.path.join(output_folder, filename)

            with open(filepath, 'wb') as f:
                f.write(download_r.content)

            print("Downloaded and saved:", filename)
            print("File size:", os.path.getsize(filepath), "bytes")
            any_saved_for_row = True

        except requests.exceptions.RequestException as e:
            error_str = f"Failed to download from {link}: {str(e)}"
            print(error_str)
            errors_for_row.append(error_str)
            n_fails += 1
            # fail: do not change any_saved_for_row, no increment

    # after trying all links for this row, update saved column
    df.at[index, 'saved'] += 1 if any_saved_for_row else 0
    if errors_for_row:
        # Join errors separated by ';' or keep as list if preferred
        df_copy.at[index, 'error_type'] = "; ".join(errors_for_row)

print(f"Failed to download from {n_fails} links")

[0] Processing: Action against child trafficking for labour and sexual exploitation in Peru – phase II - Final evaluation, 11673
File already exists, skipping: 11673_1.pdf
File already exists, skipping: 11673_2.pdf
File already exists, skipping: 11673_3.pdf
File already exists, skipping: 11673_4.pdf
[1] Processing: Thematic evaluation on IFAD's support to gender equality and women's empowerment, 13752
Failed to download from https://ioe.ifad.org/en/w/thematic-evaluation-on-ifad-s-support-to-gender-equality-and-empowerment?p_l_back_url=%2Fen%2Fthematic-evaluations&p_l_back_url_title=Thematic+Evaluations: 403 Client Error: Forbidden for url: https://ioe.ifad.org/en/w/thematic-evaluation-on-ifad-s-support-to-gender-equality-and-empowerment?p_l_back_url=%2Fen%2Fthematic-evaluations&p_l_back_url_title=Thematic+Evaluations
[2] Processing: Terminal Evaluation of the UNEP Project “Secretariat Services to the Strategic Approach to International Chemicals Management - PIMS 02005” (2018-2024), 15

In [22]:
def extract_ext(url_ext: str, content_type: str) -> str:
    ext = "" # default
    if "pdf" in content_type:
        ext = ".pdf"
    elif (
                "msword" in content_type or               # .doc
                "wordprocessingml.document" in content_type or  # .docx
                "application/doc" in content_type or      # sometimes used for .doc
                "application/vnd.ms-word" in content_type or
                "application/vnd.msword" in content_type or
                "application/vnd.openxmlformats-officedocument.wordprocessingml.document" in content_type
            ):
                # choose .docx for newer formats, .doc for older 
        if "openxmlformats-officedocument" in content_type or "wordprocessingml.document" in content_type:
            ext = ".docx"
        else:
            ext = ".doc"
    elif "zip" in content_type:
        ext = ".zip"
    elif "octet-stream" in content_type:
        if url_ext.lower() in [".pdf", ".doc", ".docx", ".zip"]:
            ext = url_ext.lower()
    else:
        if url_ext.lower() in [".pdf", ".doc", ".docx", ".zip"]:
            ext = url_ext.lower()
    return ext

In [17]:
df_copy.to_csv("member_publications_afterdownload.csv")

In [ ]:
df_copy['error_type'].value_counts()
# 994 NA links -- not my problem -- but let's still check whether there are really no links
# most of the rest of them i think are handled by the below

error_type
Failed to download from NA: Invalid URL 'NA': No scheme supplied. Perhaps you meant https://NA?                                                                                                                                                                                                             994
Failed to download from https://www.wfp.org/publications/evaluation-philippines-wfp-country-strategic-plan-2018-2023: 403 Client Error: Forbidden for url: https://www.wfp.org/publications/evaluation-philippines-wfp-country-strategic-plan-2018-2023                                                       3
Failed to download from https://www.wfp.org/publications/evaluation-madagascar-wfp-country-strategic-plan-2019-2023: 403 Client Error: Forbidden for url: https://www.wfp.org/publications/evaluation-madagascar-wfp-country-strategic-plan-2019-2023                                                         3
Failed to download from https://www.wfp.org/publications/evaluation-bhutan-wf

### Gap-filling part (messy)

In [13]:
df_nm = pd.read_csv("member_publications_afterdownload.csv")
df_nm = df_nm[~pd.isna(df_nm['error_type'])]
df_nm.head()

,Unnamed: 0,title,agency,link,download-link,type,themes,geo-scope,countries,SDGs,item-ID,saved,error_type
1,1,Thematic evaluation on IFAD's support to gende...,International Fund for Agricultural Development,https://www.unevaluation.org/member_publicatio...,https://ioe.ifad.org/en/w/thematic-evaluation-...,Global/Thematic,Gender,NaN,NaN,NaN,13752,0,Failed to download from https://ioe.ifad.org/e...
2,2,Terminal Evaluation of the UNEP Project “Secre...,United Nations Environment Programme,https://www.unevaluation.org/member_publicatio...,https://wedocs.unep.org/20.500.11822/47207,Project/Programme,NaN,Global,NaN,NaN,15075,0,Failed to download from https://wedocs.unep.or...
3,3,Evaluation of WFP's Environmental Policy,World Food Programme,https://www.unevaluation.org/member_publicatio...,https://www.wfp.org/publications/evaluation-wf...,Policy/Strategy,Environment,Global,NaN,SDG2 - Zero Hunger,11634,0,Failed to download from https://www.wfp.org/pu...
4,4,Evaluation of Global Health Days 2019-2023,World Health Organization,https://www.unevaluation.org/member_publicatio...,NaN,Global/Thematic,Communications and media,Global,NaN,SDG3 - Good Health and Well-Being,15542,0,Failed to download from NA: Invalid URL 'NA': ...
7,7,WIPO Development Agenda Coordination Division ...,World Intellectual Property Organization,https://www.unevaluation.org/member_publicatio...,NaN,Evaluability Assessment,Governance,NaN,NaN,NaN,15518,0,Failed to download from NA: Invalid URL 'NA': ...


In [ ]:
# wfp.org -- complete (but we only get links for half of them, could look at the structure of the rest)

# wedocs.unep.org -- complete

# ioe.ifad.org [1]

# unevaluation.org [20]

# evaluationreports.unicef.org [24]

# erc.undp.org [57]

NameError: name 'df' is not defined

In [ ]:
import pandas as pd
df_nm = pd.read_csv("not_matched_evaluations.csv")
# 490 missing documents after filtering NA download links 

# note that some pdf file names might be imperfectly matched to the local database

In [8]:
df_nm['saved'] = 0

In [ ]:
# count different "storage" links for documents 
import numpy as np
# df_nm['download-link'] = df_nm['download-link'].astype(str)
df_nm['download-link-indirect'] = np.nan

# wfp_count = 0
# wedocs_count = 0
# ioe_count = 0
# unevaluation_count = 0
# evaluationreports_count = 0
# erc_count = 0

import requests
from rich.pretty import pprint

for index,link in enumerate(df_nm['download-link']):
    print(f'Fetching document {index}')
    if "wfp.org" in link:

        # extract download links here following link as url

        df_nm.at[index, 'download-link-indirect'] = download_links
        print(f'Download from {link}')
        # wfp_count += 1
    # elif "wedocs.unep.org" in link:
    #     print(f'Download from {link}')
    #     # wedocs_count += 1
    # elif "ioe.ifad.org" in link:
    #     print(f'Download from {link}')
    #     # ioe_count += 1
    # elif "unevaluation.org" in link:
    #     print(f'Download from {link}')
    #     # unevaluation_count += 1
    # elif "evaluationreports.unicef.org" in link:
    #     print(f'Download from {link}')
    #     # evaluationreports_count += 1
    # elif "erc.undp.org" in link:
    #     print(f'Download from {link}')
    #     # erc_count += 1

Fetching document 0
Fetching document 1
Download from https://wedocs.unep.org/20.500.11822/47407
Fetching document 2
Download from https://www.wfp.org/publications/evaluation-bhutan-wfp-country-strategic-plan-2019-2023
Fetching document 3
Download from https://www.wfp.org/publications/evaluation-philippines-wfp-country-strategic-plan-2018-2023
Fetching document 4
Download from https://www.wfp.org/publications/strategic-evaluation-wfps-protection-sexual-exploitation-and-abuse
Fetching document 5
Download from https://www.wfp.org/publications/evaluation-guinea-wfp-interim-country-strategic-plan-2019-2024
Fetching document 6
Download from https://www.wfp.org/publications/evaluation-cuba-wfp-country-strategic-plan-2021-2024
Fetching document 7
Download from https://erc.undp.org/evaluation/documents/download/24101
Fetching document 8
Download from https://www.wfp.org/publications/evaluation-madagascar-wfp-country-strategic-plan-2019-2023
Fetching document 9
Download from https://www.wfp.org

In [15]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from time import sleep
from rich.pretty import pprint
import numpy as np

df_nm['download-link-indirect'] = np.nan

# set up headless Selenium only once
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

for index, link in enumerate(df_nm['download-link']):

    if pd.isna(link):
        continue  # skip empty links for the moment
    
    link = str(link).strip()

    # only process wfp.org links
    if "wfp.org" in link:
        print(f'Finding download links for document {index}')
        try:
            driver.get(link)
            time.sleep(2)  # wait for JS to load; adjust as needed

            soup = BeautifulSoup(driver.page_source, 'html.parser')

            # extract download links from download buttons
            download_links = []
            for div in soup.find_all("div", class_="document-download__buttons"):
                for a_tag in div.find_all("a", href=True):
                    #if "/download/" in a_tag["href"]:
                    download_links.append(a_tag["href"])

            df_nm.at[index, 'download-link-indirect'] = ", ".join(download_links)
            print(f'Found download links: {len(download_links)}')

        except Exception as e:
            print(f'Failed to fetch from {link}: {e}')
            df_nm.at[index, 'download-link-indirect'] = ""
    # process "wedocs.unep.org links

    elif "wedocs.unep.org" in link:
        print(f'Finding download links for document {index}')

        try:
            driver.get(link)
            time.sleep(2)  # wait for JS to load; adjust as needed

            soup = BeautifulSoup(driver.page_source, 'html.parser')

            # extract download links from download buttons
            download_links = []
            for a_tag in soup.find_all("a", class_="btn btn-warning text-uppercase"):
                if "https://wedocs.unep.org" not in a_tag["href"]:
                    download_link = "https://wedocs.unep.org" + a_tag["href"]
                    download_links.append(download_link)
                else:
                    download_links.append(a_tag["href"])
            
            df_nm.at[index, 'download-link-indirect'] = ", ".join(download_links)
            print(f'Found download links: {len(download_links)}')

        except Exception as e:
            print(f'Failed to fetch from {link}: {e}')
            df_nm.at[index, 'download-link-indirect'] = ""

driver.quit()


Finding download links for document 1


C:\Users\dand\AppData\Local\Temp\ipykernel_9748\691869123.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_nm.at[index, 'download-link-indirect'] = ", ".join(download_links)


Found download links: 0
Finding download links for document 2
Found download links: 2
Finding download links for document 7
Found download links: 2
Finding download links for document 10
Found download links: 2
Finding download links for document 12
Found download links: 0
Finding download links for document 14
Found download links: 0
Finding download links for document 15
Found download links: 0
Finding download links for document 16
Found download links: 0
Finding download links for document 22
Found download links: 0
Finding download links for document 23
Found download links: 0
Finding download links for document 25
Found download links: 0
Finding download links for document 27
Found download links: 2
Finding download links for document 28
Found download links: 0
Finding download links for document 31
Found download links: 0
Finding download links for document 32
Found download links: 1
Finding download links for document 35
Found download links: 0
Finding download links for docume

In [ ]:
# download link scraper for wfp.org
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

# set up selenium webDriver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run headless Chrome
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://www.wfp.org/publications/evaluation-bhutan-wfp-country-strategic-plan-2019-2023" 
driver.get(url)

# wait for JS to render 
time.sleep(5)

# Get page source after JS has rendered
soup = BeautifulSoup(driver.page_source, 'html.parser')

# extract download links
download_links = []
for div in soup.find_all("div", class_="document-download__buttons"):
    for a_tag in div.find_all("a", href=True):
        download_links.append(a_tag["href"])

# extract additional download links from the table (briefs, infographics, etc.) -- try this if len(download_links) == 0
for div in soup.find_all("div", class_="document-links-table__actions-cell"):
    for a_tag in div.find_all("a", href=True):
        download_links.append(a_tag["href"])    


print(download_links)

driver.quit()

['https://docs.wfp.org/api/documents/WFP-0000154963/download/', 'https://docs.wfp.org/api/documents/WFP-0000154964/download/']


In [ ]:
# download link scraper for wedocs.unep.org

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

# set up selenium webDriver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run headless Chrome
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://wedocs.unep.org/handle/20.500.11822/47407;jsessionid=14E51ABFC4927F2393783C775895753E" 
driver.get(url)

# wait for JS to render 
time.sleep(2)

# Get page source after JS has rendered
soup = BeautifulSoup(driver.page_source, 'html.parser')

# extract download links
download_links = []
for a_tag in soup.find_all("a", class_="btn btn-warning text-uppercase"):
    if "https://wedocs.unep.org" not in a_tag["href"]:
        download_link = "https://wedocs.unep.org" + a_tag["href"]
        download_links.append(download_link)
    else:
        download_links.append(a_tag["href"])

print(download_links)

driver.quit()

['https://wedocs.unep.org/bitstream/handle/20.500.11822/47407/4940_2025_te_unep_gef_sphpe_fsp_africa_Wesern%20Indian%20Ocean%20%28WIO-SAP%29.pdf?sequence=1&isAllowed=y']


In [28]:
import os
import requests
import re
from urllib.parse import urlparse

output_folder = 'SDG-synthesis-coalition-tidy-ts'
os.makedirs(output_folder, exist_ok=True)

df_nm['download-link-indirect'] = df_nm['download-link-indirect'].astype(str)
df_nm['ts_saved'] = 0
df_nm['ts_error_type'] = np.nan
df_nm = df_nm[~pd.isna(df_nm['item-ID'])]
n_fails = 0

for idx, row in df_nm.iterrows():
    links = [l.strip() for l in row['download-link-indirect'].split(";") if l.strip()]
    #title = df_nm['title'].iloc[index]
    item_id = str(int(row['item-ID']))
    print(f"[{idx}] Processing: {item_id}")

    # track if any file saved for this row
    any_saved_for_row = False
    errors_for_row = []

    for i, link in enumerate(links, start=1):
        # determine filename early to check if exists
        # we'll assume the extension is pdf by default (better to do it after requests, but for skipping, we'll try guessing from URL)
        ext = ".pdf"
        parsed_url = urlparse(link)
        _, url_ext = os.path.splitext(parsed_url.path)
        if url_ext.lower() in [".pdf", ".doc", ".docx", ".zip"]:
            ext = url_ext.lower()

        filename = f"{item_id}_{i}{ext}"
        filepath = os.path.join(output_folder, filename)

        if os.path.exists(filepath):
            print(f"File already exists, skipping: {filename}")
            any_saved_for_row = True  # count this as saved since file is already there
            continue

        try:
            download_r = requests.get(link, allow_redirects=True, timeout=30)
            download_r.raise_for_status()

            content_type = download_r.headers.get("Content-Type", "").lower()
            ext = extract_ext(url_ext, content_type)

            # rebuild filename and filepath with final extension
            #filename = f"{sanitise_filename(title)}_{i}{ext}"

            filename = f"{item_id}_{i}{ext}"
            filepath = os.path.join(output_folder, filename)

            with open(filepath, 'wb') as f:
                f.write(download_r.content)

            print("Downloaded and saved:", filename)
            print("File size:", os.path.getsize(filepath), "bytes")
            any_saved_for_row = True

        except requests.exceptions.RequestException as e:
            error_str = f"Failed to download from {link}: {str(e)}"
            print(error_str)
            errors_for_row.append(error_str)
            n_fails += 1

    # after trying all links for this row, update saved column
    df_nm.at[idx, 'ts_saved'] += 1 if any_saved_for_row else 0
    if errors_for_row:
        df_nm.at[idx, 'ts_error_type'] = "; ".join(errors_for_row)

print(f"Failed to download from {n_fails} links")

[1] Processing: 13752
[2] Processing: 15075
Failed to download from https://docs.wfp.org/api/documents/WFP-0000164009/download/, https://docs.wfp.org/api/documents/WFP-0000164010/download/: 404 Client Error: Not Found for url: https://docs.wfp.org/api/documents/WFP-0000164009/download/,%20https://docs.wfp.org/api/documents/WFP-0000164010/download/


C:\Users\dand\AppData\Local\Temp\ipykernel_9748\2154782102.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Failed to download from https://docs.wfp.org/api/documents/WFP-0000164009/download/, https://docs.wfp.org/api/documents/WFP-0000164010/download/: 404 Client Error: Not Found for url: https://docs.wfp.org/api/documents/WFP-0000164009/download/,%20https://docs.wfp.org/api/documents/WFP-0000164010/download/' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_nm.at[idx, 'ts_error_type'] = "; ".join(errors_for_row)


[3] Processing: 11634
Failed to download from nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[4] Processing: 15542
Failed to download from nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[7] Processing: 15518
Failed to download from https://docs.wfp.org/api/documents/WFP-0000164005/download/, https://docs.wfp.org/api/documents/WFP-0000164006/download/: 404 Client Error: Not Found for url: https://docs.wfp.org/api/documents/WFP-0000164005/download/,%20https://docs.wfp.org/api/documents/WFP-0000164006/download/
[9] Processing: 15546
Failed to download from nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[20] Processing: 13756
Failed to download from nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[22] Processing: 11633
[24] Processing: 11641
Failed to download from nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[25] Processing: 11637
[29] Processing: 15

In [29]:
df_nm.to_csv(f"member_publications_ts-{today}.csv")